# LLM Agora Demo
Interactively run a simple Agora session with configurable agents, LLM backends, and turn limits.

## Instructions
- Ensure `.env` defines `OPENROUTER_API_KEY`.
- Tweak `agent_configs` / `turns_per_agent` (and snapshot flags) to explore different models, prompts, and persistence paths.
- Run cells sequentially to execute Agora sessions, inspect histories, and optionally save/load snapshots.


In [1]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from agora.agora import Agora
from agora.agent import Agent
from agora.llm import OpenRouterClient
from agora.persistence import load_snapshot, save_snapshot


# Purely-public debate

In [2]:
# --- Configuration ---
turns_per_agent = 2
agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-4o-mini',
        'system_prompt': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'response_instruction': 'Alpha, offer your next public utterance.',
    },
    {
        'name': 'Beta',
        'model': 'anthropic/claude-3-haiku',
        'system_prompt': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'response_instruction': 'Beta, offer your next public utterance.',
    },
]


In [3]:
# --- Build agents and run the Agora ---
client = OpenRouterClient()
try:
    agents = []
    for cfg in agent_configs:
        agent = Agent(
            name=cfg['name'],
            model=cfg['model'],
            llm_client=client,
            system_prompt=cfg.get('system_prompt', ''),
            response_instruction=cfg['response_instruction'],
        )
        agents.append(agent)

    agora = Agora(agents)
    history = agora.run(max_turns_per_agent=turns_per_agent)

    print(f'Total turns: {len(history)}')
    for turn in history:
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
finally:
    client.close()


Total turns: 4
Turn 01 | Alpha: Discover our revolutionary widgets that enhance productivity and streamline your workflow!
Turn 02 | Beta: Prove it.
Turn 03 | Alpha: Experience the efficiency of our widgets with a 30-day money-back guarantee!
Turn 04 | Beta: Show me the data.


In [4]:
# --- Inspect each agent's perspective on the history ---
for agent in agents:
    print(f"\n### History visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")



### History visible to Alpha
Turn 01 | Alpha: Discover our revolutionary widgets that enhance productivity and streamline your workflow!
Turn 02 | Beta: Prove it.
Turn 03 | Alpha: Experience the efficiency of our widgets with a 30-day money-back guarantee!
Turn 04 | Beta: Show me the data.

### History visible to Beta
Turn 01 | Alpha: Discover our revolutionary widgets that enhance productivity and streamline your workflow!
Turn 02 | Beta: Prove it.
Turn 03 | Alpha: Experience the efficiency of our widgets with a 30-day money-back guarantee!
Turn 04 | Beta: Show me the data.


# Public debate with private reflections
The option to save and/or load the state is also available

In [5]:
# --- Private reflection configuration ---
private_turns_per_agent = 2

private_snapshot_path = Path('../snapshots/private_snapshot.json')
load_private_snapshot = False  # Set True to resume from a saved snapshot.
save_private_snapshot = False  # Set True to persist the new session.

private_agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-4o-mini',
        'system_prompt': 'You are Alpha, a seller of a widget who speaks in concise, single sentences. Beta is your potential buyer. Gamma is in the audience.',
        'response_instruction': 'Alpha, offer your next public utterance.',
        'private_response_instruction': 'Alpha, what are you thinking privately?',
    },
    {
        'name': 'Beta',
        'model': 'anthropic/claude-3-haiku',
        'system_prompt': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences. Alpha is your salesman. Gamma is in the audience.',
        'response_instruction': 'Beta, offer your next public utterance.',
        'private_response_instruction': 'Beta, what are you thinking privately?',
    },
    {
        'name': 'Gamma',
        'model': 'openai/gpt-4o-mini',
        'system_prompt': 'You are Gamma, listening to the conversation between the salesman, Alpha, and the potential buyer, Beta. You helpfully ad-lib their conversation in concise, single sentences.',
        'response_instruction': 'Gamma, offer your next public utterance.',
        'private_response_instruction': 'Gamma, what are you thinking privately?',
    },
]


In [6]:
# --- Run Agora with private reflections ---
private_agents = []
private_client = OpenRouterClient()
try:
    if load_private_snapshot and private_snapshot_path.exists():
        def _snapshot_factory(state):
            return private_client
        private_agora = load_snapshot(private_snapshot_path, _snapshot_factory)
        private_agents = list(private_agora.agents)
        print(f'Loaded snapshot from {private_snapshot_path}')
    else:
        for cfg in private_agent_configs:
            agent = Agent(
                name=cfg['name'],
                model=cfg['model'],
                llm_client=private_client,
                system_prompt=cfg.get('system_prompt', ''),
                response_instruction=cfg['response_instruction'],
                private_response_instruction=cfg.get('private_response_instruction'),
            )
            private_agents.append(agent)
        private_agora = Agora(private_agents)

    private_history = private_agora.run(max_turns_per_agent=private_turns_per_agent)

    print(f'Total turns (including private reflections): {len(private_history)}')
    for turn in private_history:
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private): {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
finally:
    private_client.close()


Total turns (including private reflections): 12
Turn 01 | Alpha (private): I'm focused on making a sale and showcasing the benefits of my widget.
Turn 02 | Alpha: Our widget is designed to simplify your daily tasks and increase efficiency.
Turn 03 | Beta (private): Beta: Hmm, sounds like a sales pitch.
Turn 04 | Beta: I'm not convinced. What specific benefits can the widget provide?
Turn 05 | Gamma (private): I'm considering how Alpha can highlight unique features of the widget and provide real-world examples to demonstrate its value.
Turn 06 | Gamma: Alpha: The widget streamlines communication, automates repetitive tasks, and saves you up to two hours a day, allowing you to focus on what matters most.
Turn 07 | Alpha (private): I'm confident that highlighting the time-saving aspect will resonate with Beta's needs.
Turn 08 | Alpha: Additionally, our widget is user-friendly and requires no technical expertise to operate.
Turn 09 | Beta (private): Beta: Saves two hours a day, you say? Pr

In [7]:
# --- Save private snapshot (optional) ---
if save_private_snapshot:
    save_snapshot(private_snapshot_path, private_agora)
    print(f'Snapshot saved to {private_snapshot_path}')
else:
    print('Snapshot not saved (set save_private_snapshot=True to persist).')


Snapshot not saved (set save_private_snapshot=True to persist).


In [8]:
# --- Inspect each agent's view (private reflections stay local) ---
for agent in private_agents:
    print(f"\n### Full history visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private): {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")



### Full history visible to Alpha
Turn 01 | Alpha (private): I'm focused on making a sale and showcasing the benefits of my widget.
Turn 02 | Alpha: Our widget is designed to simplify your daily tasks and increase efficiency.
Turn 04 | Beta: I'm not convinced. What specific benefits can the widget provide?
Turn 06 | Gamma: Alpha: The widget streamlines communication, automates repetitive tasks, and saves you up to two hours a day, allowing you to focus on what matters most.
Turn 07 | Alpha (private): I'm confident that highlighting the time-saving aspect will resonate with Beta's needs.
Turn 08 | Alpha: Additionally, our widget is user-friendly and requires no technical expertise to operate.
Turn 10 | Beta: Beta: That's a bold claim. I'll need to see a demonstration before I consider purchasing.
Turn 12 | Gamma: Alpha: Absolutely! Let's schedule a demonstration so you can see the widget in action and experience its benefits firsthand.

### Full history visible to Beta
Turn 02 | Alpha: